In [47]:
from zebrafish_ms2_paper.utils import pboc_rc, style_axes, colors, fontsize
from zebrafish_ms2_paper.gillespie_simulations_delay import Params, hill_function
import matplotlib.pyplot as plt
from matplotlib import rc, rcParams
import numpy as np

In [48]:
rcParams.update(pboc_rc)
rcParams['pdf.fonttype'] = 42
fontsize = 9

In [49]:
colors

{'green': '#7AA974',
 'light_green': '#BFD598',
 'pale_green': '#DCECCB',
 'yellow': '#EAC264',
 'light_yellow': '#F3DAA9',
 'pale_yellow': '#FFEDCE',
 'blue': '#738FC1',
 'light_blue': '#A9BFE3',
 'pale_blue': '#C9D7EE',
 'red': '#D56C55',
 'light_red': '#E8B19D',
 'pale_red': '#F1D4C9',
 'purple': '#AB85AC',
 'light_purple': '#D4C2D9',
 'dark_green': '#7E9D90',
 'dark_brown': '#905426'}

In [50]:
"""solve the deterministic DDE using the Euler method"""

def dde_model_derivatives(mrna, protein, delayed_protein, p):
    dmrna_dt = p.transcription_rate_1 / (1 + (delayed_protein / p.KD_transcription_rate) ** p.n) - p.mrna_decay_rate * mrna
    dprotein_dt = p.translation_rate * mrna - p.protein_decay_rate * protein
    
    return dmrna_dt, dprotein_dt

def solve_dde(p):
    t_arr = np.arange(0, p.Tmax, p.dt)
    delay_index = int(np.round(p.delay / p.dt))
    mrna = np.zeros_like(t_arr)
    protein = np.zeros_like(t_arr)
    mrna[0] = p.initial_mrna
    protein[0] = p.initial_protein
    
    for i in range(1, len(t_arr)):
        delayed_index = int(np.clip(i - delay_index, a_min=0, a_max=np.inf))
        delayed_protein = protein[delayed_index]
        
        dmrna_dt, dprotein_dt = dde_model_derivatives(mrna[i - 1], protein[i - 1], delayed_protein, p)
        mrna[i] = mrna[i - 1] + p.dt * dmrna_dt
        protein[i] = protein[i - 1] + p.dt * dprotein_dt
        
    return mrna, protein, t_arr

In [70]:
%matplotlib qt

p = Params()
p.initial_state = np.array([1])
p.Tmax = 120 + 90
p.k_off0 = 0.0
p.k_off1 = 0.0
p.k_on0 = 0.0
p.k_on1 = 0.0
p.transcription_rate_0 = 0

p.translation_rate = 4.5
p.transcription_rate_1 = 10.0 / len(p.initial_state)

p.mrna_decay_rate = 0.23
p.protein_decay_rate = 0.23

p.delay = 7.5

p.KD_transcription_rate = 100
p.dt = 0.01

# sharp response
p.n = 20
mrna, protein, t_arr = solve_dde(p)

delay_index = int(np.round(p.delay) / p.dt)
delayed_protein = np.roll(protein, delay_index)
burn_in_time = 90
mrna = mrna[t_arr > burn_in_time]
protein = protein[t_arr > burn_in_time]
delayed_protein = delayed_protein[t_arr > burn_in_time]
t_arr = t_arr[t_arr > burn_in_time] - burn_in_time

production_rate_sharp = p.transcription_rate_1 * hill_function(delayed_protein, p.KD_transcription_rate, p.n)
protein_sharp = protein
t_arr_sharp = t_arr

# gradual response
p.n = 3
mrna, protein, t_arr = solve_dde(p)

delay_index = int(np.round(p.delay) / p.dt)
delayed_protein = np.roll(protein, delay_index)
burn_in_time = 90
mrna = mrna[t_arr > burn_in_time]
protein = protein[t_arr > burn_in_time]
delayed_protein = delayed_protein[t_arr > burn_in_time]
t_arr = t_arr[t_arr > burn_in_time] - burn_in_time

production_rate_gradual = p.transcription_rate_1 * hill_function(delayed_protein, p.KD_transcription_rate, p.n)
protein_gradual = protein
t_arr_gradual = t_arr

"""plot"""
fig, axs = plt.subplots(2, 2)
ax = axs[0, 0]
ax.plot(t_arr_sharp, production_rate_sharp / np.max(production_rate_sharp), color='k')
ax.set_xlabel('time (min)', fontsize=fontsize)
ax.set_ylabel('transcription rate (a.u./min)', fontsize=fontsize)
ax = style_axes(ax, fontsize=fontsize)

ax = axs[1, 0]
ax.plot(t_arr_sharp, protein_sharp / np.max(protein_sharp), color=colors['green'])
ax.set_xlabel('time (min)', fontsize=fontsize)
ax.set_ylabel('protein concentration (a.u.)', fontsize=fontsize)
ax = style_axes(ax, fontsize=fontsize)

ax = axs[0, 1]
ax.plot(t_arr_gradual, production_rate_gradual / np.max(production_rate_gradual), color='k')
ax.set_xlabel('time (min)', fontsize=fontsize)
ax.set_ylabel('transcription rate (a.u./min)', fontsize=fontsize)
ax = style_axes(ax, fontsize=fontsize)

ax = axs[1, 1]
ax.plot(t_arr_gradual, protein_gradual / np.max(protein_gradual), color=colors['green'])
ax.set_xlabel('time (min)', fontsize=fontsize)
ax.set_ylabel('protein concentration (a.u.)', fontsize=fontsize)
ax = style_axes(ax, fontsize=fontsize)

fig.tight_layout()

In [46]:
plt.savefig(r'/home/brandon/Documents/Code/zebrafish-ms2-paper/figures/square_vs_sine_sim_fig/sharp_vs_gradual_regulation.png')

## Plot input-output functions

In [72]:
x = np.logspace(1, 3, 1000)

y1 = hill_function(x, p.KD_transcription_rate, 3)
y2 = hill_function(x, p.KD_transcription_rate, 20)

plt.figure(figsize=(4,3))
plt.plot(x, y1, color=colors['red'], label='$n=3$')
plt.plot(x, y2, color=colors['blue'], label='$n=20$')
plt.xscale('log')
plt.xlabel('protein concentration (a.u.)', fontsize=fontsize)
plt.ylabel('transcription rate (a.u./min)', fontsize=fontsize)
plt.legend(fontsize=fontsize, facecolor='w')
ax = style_axes(plt.gca(), fontsize=fontsize)
ax.minorticks_off()
plt.tight_layout()


In [68]:
plt.savefig(r'/home/brandon/Documents/Code/zebrafish-ms2-paper/figures/square_vs_sine_sim_fig/sharp_vs_gradual_input-output_function.pdf')

In [55]:
plt.tight_layout()

In [63]:
ax.minorticks_off()

In [69]:
fontsize

9

In [83]:
%matplotlib qt

p = Params()
p.initial_state = np.array([1])
p.Tmax = 120 + 90
p.k_off0 = 0.0
p.k_off1 = 0.0
p.k_on0 = 0.0
p.k_on1 = 0.0
p.transcription_rate_0 = 0

p.translation_rate = 4.5
p.transcription_rate_1 = 10.0 / len(p.initial_state)

p.mrna_decay_rate = 0.23
p.protein_decay_rate = 0.23

p.delay = 7.5

p.KD_transcription_rate = 100
p.dt = 0.01

# sharp response
p.n = 20
mrna, protein, t_arr = solve_dde(p)

delay_index = int(np.round(p.delay) / p.dt)
delayed_protein = np.roll(protein, delay_index)
burn_in_time = 90
mrna = mrna[t_arr > burn_in_time]
protein = protein[t_arr > burn_in_time]
delayed_protein = delayed_protein[t_arr > burn_in_time]
t_arr = t_arr[t_arr > burn_in_time] - burn_in_time

production_rate_sharp = p.transcription_rate_1 * hill_function(delayed_protein, p.KD_transcription_rate, p.n)
protein_sharp = protein
t_arr_sharp = t_arr

# gradual response
p.n = 3
mrna, protein, t_arr = solve_dde(p)

delay_index = int(np.round(p.delay) / p.dt)
delayed_protein = np.roll(protein, delay_index)
burn_in_time = 90
mrna = mrna[t_arr > burn_in_time]
protein = protein[t_arr > burn_in_time]
delayed_protein = delayed_protein[t_arr > burn_in_time]
t_arr = t_arr[t_arr > burn_in_time] - burn_in_time

production_rate_gradual = p.transcription_rate_1 * hill_function(delayed_protein, p.KD_transcription_rate, p.n)
protein_gradual = protein
t_arr_gradual = t_arr

"""plot"""
fig, axs = plt.subplots(3, 2, figsize=(6.5, 7.5))

ax = axs[0, 0]
ax.set_xticks([])
ax.set_yticks([])

# input-output function
x = np.logspace(1, 3, 1000)

y1 = hill_function(x, p.KD_transcription_rate, 3)
y2 = hill_function(x, p.KD_transcription_rate, 20)
ax = axs[0, 1]
ax.plot(x, y1, color=colors['green'], label='$n=3$')
ax.plot(x, y2, color='k', label='$n=20$')
ax.set_xscale('log')
ax.set_xlabel('protein concentration (a.u.)', fontsize=fontsize)
ax.set_ylabel('transcription rate (a.u./min)', fontsize=fontsize)
ax.legend(fontsize=fontsize, facecolor='w')
ax = style_axes(ax, fontsize=fontsize)
ax.minorticks_off()

# traces
ax = axs[1, 0]
ax.plot(t_arr_sharp, production_rate_sharp / np.max(production_rate_sharp), color=colors['blue'])
ax.set_xlabel('time (min)', fontsize=fontsize)
ax.set_ylabel('transcription rate (a.u./min)', fontsize=fontsize)
ax = style_axes(ax, fontsize=fontsize)

ax = axs[2, 0]
ax.plot(t_arr_sharp, protein_sharp / np.max(protein_sharp), color=colors['red'])
ax.set_xlabel('time (min)', fontsize=fontsize)
ax.set_ylabel('protein concentration (a.u.)', fontsize=fontsize)
ax = style_axes(ax, fontsize=fontsize)

ax = axs[1, 1]
ax.plot(t_arr_gradual, production_rate_gradual / np.max(production_rate_gradual), color=colors['blue'])
ax.set_xlabel('time (min)', fontsize=fontsize)
ax.set_ylabel('transcription rate (a.u./min)', fontsize=fontsize)
ax = style_axes(ax, fontsize=fontsize)

ax = axs[2, 1]
ax.plot(t_arr_gradual, protein_gradual / np.max(protein_gradual), color=colors['red'])
ax.set_xlabel('time (min)', fontsize=fontsize)
ax.set_ylabel('protein concentration (a.u.)', fontsize=fontsize)
ax = style_axes(ax, fontsize=fontsize)

fig.tight_layout()

In [84]:
plt.savefig(r'/home/brandon/Documents/Code/zebrafish-ms2-paper/figures/square_vs_sine_sim_fig/sharp_vs_gradual_combined_v3.pdf')